# Gridworld: sampled episodes, exact values, and policy gradients

A fixed policy turns the finite-horizon MDP in `gridworld/model.py` into trajectories.
We will inspect action probabilities, draw one complete episode, and separate
policy sampling, environmental slip, and reward noise. Then we will compute
exact values by averaging over every possible outcome under the same fixed
policy and differentiate its expected return with respect to the logits. Finally,
we will estimate that gradient from complete episodes with full-return REINFORCE.
There is no training here.

From `rl_math`, run `uv sync --locked` and then
`uv run --locked jupyter lab notebooks/01_gridworld.ipynb`. The project
environment includes NumPy, JupyterLab, and its Python kernel.
PyTorch handles differentiable probability/value calculations; NumPy owns the seeded random draws. Run
the cells in order from the repository root, `rl_math`, or this notebook's directory.

This notebook explicitly uses CUDA. The setup cell prints the GPU name;
logits, probabilities, exact values, and autograd derivatives stay on that device.
The library also supports `device="cpu"` for reference checks.


In [1]:
from pathlib import Path
import numpy as np
import torch
import platform
import sys

project = next(p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "rl_math")
               if (p / "gridworld" / "model.py").is_file())
sys.path.insert(0, str(project))
from gridworld.model import Action, Gridworld, State
from common.returns import discounted_returns
from common.policy import Policy
from gridworld.policy import TabularSoftmaxPolicy
from gridworld.sampling import sample_episode
from gridworld.values import evaluate_policy
from gridworld.gradients import differentiate_policy
from algorithms.reinforce import ReinforceEpisode, full_return_gradient

print("Python", platform.python_version(), "NumPy", np.__version__, "PyTorch", torch.__version__)

device = torch.device("cuda")
assert torch.cuda.is_available(), "CUDA is unavailable; check the NVIDIA driver and PyTorch environment."
print("Device:", device, "GPU:", torch.cuda.get_device_name(0))


Python 3.10.12 NumPy 2.2.6 PyTorch 2.14.1+cu130
Device: cuda GPU: NVIDIA GeForce RTX 3090


## The policy interface

`common/policy.py` defines `Policy[Observation]`, an abstract interface for
discrete-action policies. Its required method, `probabilities(observation)`,
returns a normalized PyTorch tensor in the environment's action order.
The tensor retains the computation graph when its probabilities depend on parameters.
The observation type can be a gridworld state or another representation,
such as a vector. `gridworld/policy.py` supplies
`TabularSoftmaxPolicy(Policy[State])`, the gridworld implementation. A rule-based policy can implement the same
method without having logits at all.

`gridworld.sampling.sample_episode` accepts `Policy[State]` and owns the random draws. It
detaches the probability tensor before passing it to NumPy's random generator.
It does not inspect the concrete policy's parameters. This keeps environment
sampling separate from how the policy represents action probabilities
and from the algorithm that may later update its parameters.

`log_probabilities(observation)` retains the same graph for score derivatives.
The default takes the log of the probabilities; the tabular policy overrides
it with `torch.log_softmax` to avoid underflow when logits are far apart.


## One row of a tabular policy

For the full state $s=(\mathrm{cell},\mathrm{remaining})$, assign four logits
$\theta_{s,a}$ in **north, east, south, west** order:

$$\pi_\theta(a\mid s)=\frac{\exp(\theta_{s,a})}{\sum_b\exp(\theta_{s,b})}.$$

The row $(\log 1,\log 2,\log 3,\log 4)$ gives probabilities
$(0.1,0.2,0.3,0.4)$. `probabilities` uses `torch.softmax`, which evaluates the ratios stably and
records their derivatives. Each logit row is an owned float64 CUDA
`torch.nn.Parameter`; changing the caller's original array/tensor leaves it unchanged.


In [2]:
world = Gridworld()  # Default 5x5 map; horizon=12, slip=0.2.
example = TabularSoftmaxPolicy({world.start_state: torch.tensor([1., 2., 3., 4.], dtype=torch.float64).log()}, device=device)
for action, probability in zip(Action, example.probabilities(world.start_state)):
    print(f"{action.name:5s}: {probability:.1f}")
assert torch.isclose(example.probabilities(world.start_state).sum(), example.probabilities(world.start_state).new_tensor(1.))


NORTH: 0.1
EAST : 0.2
SOUTH: 0.3
WEST : 0.4


## A fixed preference for the longer safe route

The bottom row is `SHHHG`. Going north, east four times, then south avoids
the hazards when all moves succeed. Give the preferred action logit $\log 7$
and every other action logit $0$: the policy samples its preference with
probability $0.7$ and each other action with probability $0.1$.

This is a manually specified policy. Every nonterminal cell/time pair has its
own row, although we use the same preference at a cell for each remaining time.
The policy may choose another action, and slip may change where that action
lands. Even the preferred action does not guarantee a safe trajectory.

A **tabular** policy stores separate parameters for each state rather than
sharing them through a neural network. It can learn when an optimizer
updates those logits. Here they are chosen by hand and held fixed:
collecting episodes only generates data. The gridworld experiments first
hold the policy fixed to compare gradient estimators with an exact reference;
the later CartPole experiments add learning updates.


In [3]:
logits = {}
for state in world.states():
    if world.is_terminal(state):
        continue  # No actions after the goal or the task horizon.
    row, column = state.cell
    preferred = Action.NORTH if row == 4 else Action.EAST if column < 4 else Action.SOUTH
    logits[state] = torch.zeros(len(Action), dtype=torch.float64)
    logits[state][tuple(Action).index(preferred)] = np.log(7)
policy = TabularSoftmaxPolicy(logits, device=device)
print("Start probabilities:", policy.probabilities(world.start_state).detach().cpu().numpy().round(2))


Start probabilities: [0.7 0.1 0.1 0.1]


## Draw and inspect a complete episode

At each time $t$, `sample_episode` makes three draws:

$$a_t\sim\pi_\theta(\cdot\mid s_t),\qquad
s_{t+1}\sim P(\cdot\mid s_t,a_t),\qquad
r_{t+1}=\bar r(s_t,a_t,s_{t+1})+\epsilon_{t+1},$$

where $\epsilon_{t+1}\sim\mathcal N(0,\sigma^2)$ independently at each step.
Here $\bar r$ is the selected transition's conditional expected reward;
`world.expected_reward(s, a)` additionally averages over possible next states.

Use seed **7**, slip **0.2**, horizon **12**, and reward-noise standard deviation
**2.0** for this illustration. NumPy `SeedSequence(seed).spawn(3)` creates separate
streams for actions, transitions, and reward noise; each uses `default_rng`.
The sampler leaves global random state alone. Repeating a seed replays an episode.
Use distinct nonnegative integer seeds for new samples, and keep the Python/NumPy
environment and all settings fixed when reproducing a run.
See [NumPy's stream-spawning explanation](https://numpy.org/doc/stable/reference/random/parallel.html).


In [4]:
seed, noise_std, gamma = 7, 2.0, 0.9
episode = sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
assert episode == sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
print(f"seed={episode.seed}, sigma={episode.reward_noise_std}, horizon={world.horizon}, slip={world.slip}")
print(" t   cell  left action  reward -> next cell left   end")
for step in episode.steps:
    print(f"{step.t:2d} {str(step.state.cell):>7} {step.state.remaining:4d} {step.action.name:5s} "
          f"{step.reward:7.3f} -> {str(step.next_state.cell):>7} "
          f"{step.next_state.remaining:4d} {str(step.terminated):>5}")
print("Ending:", episode.end_reason)
print(f"G_0 (gamma={gamma}) = {discounted_returns([s.reward for s in episode.steps], gamma)[0]:.3f}")


seed=7, sigma=2.0, horizon=12, slip=0.2
 t   cell  left action  reward -> next cell left   end
 0  (4, 0)   12 EAST   -5.921 ->  (4, 1)   11 False
 1  (4, 1)   11 NORTH   1.216 ->  (3, 1)   10 False
 2  (3, 1)   10 EAST   -2.025 ->  (3, 2)    9 False
 3  (3, 2)    9 SOUTH  -4.254 ->  (4, 2)    8 False
 4  (4, 2)    8 EAST   -5.756 ->  (4, 3)    7 False
 5  (4, 3)    7 NORTH   1.383 ->  (3, 3)    6 False
 6  (3, 3)    6 NORTH  -1.627 ->  (2, 3)    5 False
 7  (2, 3)    5 EAST   -2.542 ->  (3, 3)    4 False
 8  (3, 3)    4 NORTH  -0.418 ->  (2, 3)    3 False
 9  (2, 3)    3 EAST   -0.367 ->  (2, 4)    2 False
10  (2, 4)    2 SOUTH  -2.187 ->  (2, 4)    1 False
11  (2, 4)    1 SOUTH  -1.517 ->  (3, 4)    0  True
Ending: horizon
G_0 (gamma=0.9) = -16.169


In [5]:
# Mark all visited cells; the table above gives their order and revisits.
visited = {step.state.cell for step in episode.steps} | {episode.steps[-1].next_state.cell}
for row, line in enumerate(world.layout):
    print(" ".join(symbol + ("*" if (row, column) in visited else " ")
                   for column, symbol in enumerate(line)))
print("* visited; S start, G goal, H hazard, # wall")


.  .  .  .  . 
.  .  #  .  . 
.  .  #  .* .*
.  .* .* .* .*
S* H* H* H* G 
* visited; S start, G goal, H hazard, # wall


In the NumPy seed-7 run, the policy samples EAST at t=0 even though NORTH
has higher probability; this sends it onto a hazard. At t=7 it selects
EAST from (2, 3) but moves SOUTH to (3, 3): that is environmental slip.
The episode ends at the horizon before reaching the goal.

The recorded `t` starts at zero for each episode; it is the exponent in the
outer $\gamma^t$ policy-gradient weight introduced in `common/returns.py`. Every
actual move decreases remaining time by one, including a move blocked by a wall.

The final transition is marked `terminated=True`: reaching the goal or
exhausting this task's horizon gives zero future value. There is no absorbing
tail, rollout cutoff, or external time-limit truncation in this sampler.
Reaching the goal on the last allowed move still pays the goal reward.


## Why noise changes variance, not the expected reward

The logits change action probabilities and thus which states are visited.
The environment supplies slip and reward noise. With this fixed state-only
policy, reward noise does not influence future actions or transitions.

Independence and zero mean give

$$\mathbb E[r_{t+1}\mid s_t,a_t,s_{t+1}]=\bar r(s_t,a_t,s_{t+1}),$$

so averaging over next states also preserves `world.expected_reward(s, a)`.
The added conditional reward variance is $\sigma^2$. Given a state/action path
of length $T$, the noise contribution to its discounted return has mean zero
and variance

$$\operatorname{Var}\!\left[\sum_{t=0}^{T-1}\gamma^t\epsilon_{t+1}\;\middle|\;
\mathrm{path}\right]=\sigma^2\sum_{t=0}^{T-1}\gamma^{2t}.$$

Unconditionally, expected return is also unchanged; the added return variance
averages this expression over paths. A particular noisy episode can have a
higher or lower return. It need not average to zero within that episode.

Changing only the noise scale keeps the same sampled path because its random
stream is separate. Compare the clean and noisy rewards on that path below.


In [6]:
clean = sample_episode(world, policy, seed=seed)
assert [(s.state, s.action, s.next_state) for s in clean.steps] == [
    (s.state, s.action, s.next_state) for s in episode.steps]
print(" t model reward   noise observed reward")
for base, noisy in zip(clean.steps, episode.steps):
    print(f"{base.t:2d} {base.reward:12.3f} {noisy.reward - base.reward:7.3f} {noisy.reward:15.3f}")
times = np.arange(len(episode.steps))
variance = noise_std**2 * np.sum(gamma**(2 * times))
print(f"Added return variance conditional on this path: {variance:.3f}")


 t model reward   noise observed reward
 0       -6.000   0.079          -5.921
 1       -1.000   2.216           1.216
 2       -1.000  -1.025          -2.025
 3       -6.000   1.746          -4.254
 4       -6.000   0.244          -5.756
 5       -1.000   2.383           1.383
 6       -1.000  -0.627          -1.627
 7       -1.000  -1.542          -2.542
 8       -1.000   0.582          -0.418
 9       -1.000   0.633          -0.367
10       -1.000  -1.187          -2.187
11       -1.000  -0.517          -1.517
Added return variance conditional on this path: 19.373


## Exact values by backward dynamic programming

A sampled $G_t$ is one return. The value $V^\pi(s)$ is its expectation under
the fixed policy. Since $s$ contains remaining time, the tables depend on both
cell and time; discounting starts at the state being evaluated:

$$Q^\pi(s,a)=\sum_{s'}P(s'\mid s,a)
  [\bar r(s,a,s')+\gamma V^\pi(s')],\qquad
V^\pi(s)=\sum_a\pi(a\mid s)Q^\pi(s,a),\qquad
A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s).$$

`gridworld.values.evaluate_policy` starts with zero values at the goal and at
remaining time zero, then computes rows with 1, 2, ..., horizon steps left.
Every nonterminal successor has one fewer step left, so its value is already
available. The code's `torch.dot(probabilities, rewards + gamma * future_values)`
averages over model outcomes; `torch.dot(policy.probabilities(state), q[state])`
averages over actions. No samples or convergence loop are needed, even at
$\gamma=1$. All non-wall cell/time pairs are evaluated, including unreachable ones.

V, Q, advantage, and J are float64 tensors. `torch.stack` combines successor
values while retaining their graph; converting them to Python floats, NumPy,
or `torch.tensor(list_of_values)` inside this recurrence would break that path.
Fixed transition probabilities and mean rewards are constant tensors. We detach
only for display/sampling, and use `.item()` when a scalar is needed for a check.

For our deterministic start, $J(\theta)=V^{\pi_\theta}(s_0)$.
Independent zero-mean reward noise leaves this expectation unchanged.

### A two-step calculation you can check by hand

Use the one-row map `SHG`, no slip, horizon 2, $\gamma=0.5$, and
$\pi=(0.1,0.2,0.3,0.4)$ in north/east/south/west order at every nonterminal
state. An ordinary step pays $-1$, entering or staying on H pays $-6$, and
entering G pays $9$. Start with zero terminal values.

With one step left, $V(S,1)=0.8(-1)+0.2(-6)=-2$ and
$V(H,1)=0.4(-6)+0.2(9)+0.4(-1)=-1$.
With two steps left, an EAST action at S has
$Q(S,2,\mathrm{E})=-6+0.5V(H,1)=-6.5$;
the other actions stay at S, giving $-1+0.5V(S,1)=-2$.
Thus $J=V(S,2)=0.8(-2)+0.2(-6.5)=-2.9$.
The computed table below includes Q and advantage in the same action order.

In [7]:
tiny = Gridworld(layout=("SHG",), horizon=2, slip=0)
tiny_policy = TabularSoftmaxPolicy({
    s: torch.tensor([1., 2., 3., 4.], dtype=torch.float64).log() for s in tiny.states() if not tiny.is_terminal(s)
}, device=device)
tiny_values = evaluate_policy(tiny, tiny_policy, gamma=0.5)
print("cell left | Q(N,E,S,W)               | V     | A(N,E,S,W)")
for remaining in (1, 2):
    for cell, label in ((tiny.start, "S"), ((0, 1), "H")):
        state = State(cell, remaining)
        print(f"{label:>4} {remaining:4d} | {str(tiny_values.q[state].detach().cpu().numpy().round(2)):24s} | "
              f"{tiny_values.v[state]:5.2f} | {tiny_values.advantage[state].detach().cpu().numpy().round(2)}")
assert np.isclose(tiny_values.start_return.item(), -2.9)
assert all(tiny_values.v[s] == 0 for s in tiny.states() if tiny.is_terminal(s))
print(f"J = V(S,2) = {tiny_values.start_return:.2f}; all terminal V/Q/A rows are zero.")

cell left | Q(N,E,S,W)               | V     | A(N,E,S,W)
   S    1 | [-1. -6. -1. -1.]        | -2.00 | [ 1. -4.  1.  1.]
   H    1 | [-6.  9. -6. -1.]        | -1.00 | [-5. 10. -5.  0.]
   S    2 | [-2.  -6.5 -2.  -2. ]    | -2.90 | [ 0.9 -3.6  0.9  0.9]
   H    2 | [-6.5  9.  -6.5 -2. ]    | -1.60 | [-4.9 10.6 -4.9 -0.4]
J = V(S,2) = -2.90; all terminal V/Q/A rows are zero.


The policy-weighted advantage is zero because

$$\sum_a\pi(a\mid s)A^\pi(s,a)
=\sum_a\pi(a\mid s)Q^\pi(s,a)-V^\pi(s)\sum_a\pi(a\mid s)
=V^\pi(s)-V^\pi(s)=0.$$

At $(S,2)$, this is $0.8(0.9)+0.2(-3.6)=0$. The **unweighted** average
need not vanish. Positive advantage means better than the policy's average
choice at that state, even when Q itself is negative.

Replacing the policy average over next actions with a maximum chooses the
best continuation instead of following $\pi$. Applied throughout the backward
pass, that gives the Bellman optimality equation and a different policy's
return. Here even the current row's maximum Q, $-2$, differs from its
policy-weighted value, $-2.9$. We keep the policy fixed.

### Evaluate the earlier 5×5 policy

Now use the same manually chosen safe-route preferences, slip 0.2, horizon 12,
and $\gamma=0.9$ as the sampled episode. Each printed map fixes remaining time.
G has value zero because its entry reward was paid on the preceding transition;
the horizon-zero slice is also entirely zero. These are exact model
expectations, so a single noisy trajectory need not have return near J.

In [8]:
values = evaluate_policy(world, policy, gamma=gamma)
print(f"Expected start return J (gamma={gamma}): {values.start_return:.6f}")
for remaining in (1, world.horizon):
    print(f"\nV with {remaining} step(s) left:")
    for row, line in enumerate(world.layout):
        print(" ".join("    #  " if symbol == "#" else
                       f"{values.v[State((row, column), remaining)]:7.2f}"
                       for column, symbol in enumerate(line)))

nonterminal = [s for s in world.states() if not world.is_terminal(s)]
residual = max(abs(torch.dot(policy.probabilities(s), values.advantage[s]).item())
               for s in nonterminal)
assert residual < 1e-12
print(f"\nMax |sum_a pi(a|s) A(s,a)| over nonterminal rows: {residual:.2e}")

Expected start return J (gamma=0.9): -8.556035

V with 1 step(s) left:
  -1.00   -1.00   -1.00   -1.00   -1.00
  -1.00   -1.00     #     -1.00   -1.00
  -1.00   -1.00     #     -1.00   -1.00
  -1.00   -1.70   -1.70   -1.70    4.80
  -1.70   -2.40   -3.10   -1.00    0.00

V with 12 step(s) left:
  -6.50   -5.87   -4.74   -3.43   -2.19
  -7.11   -7.02     #     -1.86   -0.34
  -7.26   -7.16     #      0.09    2.35
  -7.10   -6.32   -3.41    0.76    5.94
  -8.56   -8.71   -7.07   -1.71    0.00

Max |sum_a pi(a|s) A(s,a)| over nonterminal rows: 1.39e-15


## Differentiate the exact expected return with PyTorch

Knowing the model lets us differentiate $J(\theta)=V^{\pi_\theta}(s_0)$ itself.
The evaluator has already built the graph
**logits → softmax probabilities → Bellman expectations → J**.
`gridworld.gradients.differentiate_policy` evaluates J, then calls
[`torch.autograd.grad`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html)
on it and all the logit parameters. PyTorch applies the chain rule through the
whole graph; we do not implement a separate backward algorithm.

`materialize_grads=True` returns zeros for rows disconnected from J, including
optional terminal rows. `autograd.grad` returns derivatives without accumulating
the parameters' `.grad` buffers or updating them. The returned `PolicyGradient`
contains `values` and `gradient`, a detached tensor for each supplied logit row.

For interpretation, the local softmax derivative is still

$$\frac{\partial\pi(a\mid s)}{\partial\theta_{s,b}}
=\pi(a\mid s)\bigl(\mathbf 1[a=b]-\pi(b\mid s)\bigr).$$

Combining this with the Bellman expectations gives

$$\frac{\partial J}{\partial\theta_{s,b}}
=\gamma^t\Pr(s_t=s)\,\pi(b\mid s)\,A(s,b),
\qquad t=H-\mathrm{remaining}.$$

Each cell/remaining-time row has independent parameters. The local Q/advantage
discounts rewards relative to that state; the outer $\gamma^t$ discounts back
to the episode start. Autograd obtains both from the same gamma factors in the
value computation, including dependencies through successor values. We do not
normalize by episode length or total occupancy.


### Continue the two-step `SHG` example

Keep $\gamma=0.5$, no slip, and $\pi=(0.1,0.2,0.3,0.4)$ in each row.
At $t=0$ we are at $(S,2)$ with weight 1. At $t=1$, the state is $(S,1)$
with probability 0.8 or $(H,1)$ with probability 0.2. Their reverse weights are
$0.5(0.8)=0.4$ and $0.5(0.2)=0.1$.

For example, the EAST derivative at the start is
$1(0.2)(-3.6)=-0.72$. At $(H,1)$ it is
$0.1(0.2)(10)=0.20$: the goal reward matters only on paths visiting that row,
and its effect on J is discounted once. The complete gradient rows are below.
$(H,2)$ has nonzero local action advantages but is unreachable from the start,
so all its derivatives of J are zero.


In [9]:
reference = differentiate_policy(tiny, tiny_policy, gamma=0.5)
assert reference.values.start_return.item() == tiny_values.start_return.item()
print(f"Exact J = {reference.values.start_return.item():.2f}")
print("cell left | dJ/dlogits (N,E,S,W)")
for cell, label, remaining in ((tiny.start, "S", 2), (tiny.start, "S", 1),
                               ((0, 1), "H", 1), ((0, 1), "H", 2)):
    state = State(cell, remaining)
    print(f"{label:>4} {remaining:4d} | {reference.gradient[state].cpu().numpy().round(4)}")
torch.testing.assert_close(reference.gradient[State((0, 1), 1)],
                           reference.gradient[State((0, 1), 1)].new_tensor([-.05, .2, -.15, 0]),
                           rtol=1e-12, atol=1e-14)


Exact J = -2.90
cell left | dJ/dlogits (N,E,S,W)
   S    2 | [ 0.09 -0.72  0.27  0.36]
   S    1 | [ 0.04 -0.32  0.12  0.16]
   H    1 | [-0.05  0.2  -0.15  0.  ]
   H    2 | [0. 0. 0. 0.]


### Check every logit with central finite differences

For a single parameter $\theta_{s,b}$, change only that entry and call the
previous value evaluator twice:

$$D_{s,b}(\epsilon)=\frac{J(\theta+\epsilon e_{s,b})
-J(\theta-\epsilon e_{s,b})}{2\epsilon}.$$

This check differentiates the scalar return numerically without using the
autograd result. Use $\epsilon=10^{-5}$ in float64, with absolute tolerance
$10^{-8}$ plus relative tolerance $10^{-6}$ (`torch.testing.assert_close`).
The absolute term covers derivatives near zero. Central differences have
$O(\epsilon^2)$ truncation error for this smooth objective; very small steps can
instead magnify floating-point cancellation. These settings apply to this tiny,
moderately scaled example, rather than every possible logit/reward scale.

We perturb all 16 independent logits, including unreachable rows, in the `SHG`
model. Its step costs and hazard penalties give nontrivial rewards before the
last transition. Omitting the outer discount would double the nonzero $t=1$
derivatives and fail this check. Detach and clone each row to leave `tiny_policy` fixed. We use
`torch.no_grad()` for these scalar-only evaluations; the reference computation
above needs its graph enabled.


In [10]:
epsilon = 1e-5
numerical = {s: torch.zeros_like(row) for s, row in tiny_policy.logits.items()}
with torch.no_grad():
    for state in numerical:
        for b in range(len(Action)):
            returns = []
            for sign in (1, -1):
                perturbed = {s: row.detach().clone() for s, row in tiny_policy.logits.items()}
                perturbed[state][b] += sign * epsilon
                returns.append(evaluate_policy(
                    tiny, TabularSoftmaxPolicy(perturbed, device=device), gamma=0.5
                ).start_return.item())
            numerical[state][b] = (returns[0] - returns[1]) / (2 * epsilon)

print("cell left | max |autograd - finite difference|")
for state in numerical:
    error = (reference.gradient[state] - numerical[state]).abs().max().item()
    print(f"{str(state.cell):>6} {state.remaining:4d} | {error:.3e}")
    torch.testing.assert_close(reference.gradient[state], numerical[state],
                               rtol=1e-6, atol=1e-8)
print("All 16 logit derivatives agree (epsilon=1e-5, rtol=1e-6, atol=1e-8).")


cell left | max |autograd - finite difference|
(0, 0)    1 | 2.566e-11
(0, 1)    1 | 2.097e-11
(0, 0)    2 | 4.591e-11
(0, 1)    2 | 0.000e+00
All 16 logit derivatives agree (epsilon=1e-5, rtol=1e-6, atol=1e-8).


### The exact reference for the earlier 5×5 policy

Now differentiate the same horizon-12, slip-0.2 policy at $\gamma=0.9$.
The returned J must agree with the earlier value evaluation. Each gradient row
sums to zero: adding a constant to every logit in one row changes no probabilities
and hence no return. The norm below is over all independent cell/time logits.

These are derivatives at the fixed policy, before any optimizer step. Independent
zero-mean reward noise changes neither the conditional reward means nor this
gradient, so it can later vary sampling variance while the reference stays fixed.
The unit tests also compare every derivative on a tiny **stochastic** model with
time-dependent logits and $\gamma\in\{0,0.7,1\}$, check early goal termination,
and verify constant-logit-shift invariance.


In [11]:
gradient_reference = differentiate_policy(world, policy, gamma=gamma)
assert gradient_reference.values.start_return.item() == values.start_return.item()
rows = torch.stack(list(gradient_reference.gradient.values()))
residual = rows.sum(dim=1).abs().max().item()
assert residual < 1e-12
print(f"Exact J: {gradient_reference.values.start_return.item():.6f}")
print("Start logit gradient (N,E,S,W):",
      gradient_reference.gradient[world.start_state].cpu().numpy().round(6))
print(f"Gradient L2 norm over {rows.numel()} logits: {torch.linalg.vector_norm(rows).item():.6f}")
print(f"Max |sum of logit derivatives in one row|: {residual:.2e}")


Exact J: -8.556035
Start logit gradient (N,E,S,W): [ 0.467465 -0.408304 -0.02958  -0.02958 ]


Gradient L2 norm over 1056 logits: 0.908393
Max |sum of logit derivatives in one row|: 2.22e-16


## Full-return REINFORCE: one sampled gradient

The exact reference differentiates a sum over possible futures. We can instead
use the probability of a **sampled complete trajectory**. With environment
transitions and reward noise independent of the policy parameters,

$$\nabla_\theta\log p_\theta(\tau)
=\sum_{t=0}^{T-1}\nabla_\theta\log\pi_\theta(a_t\mid s_t).$$

The log-derivative identity gives, for our discounted start-state objective,

$$J(\theta)=\mathbb E[G_0],\qquad G_0=\sum_{k=0}^{T-1}\gamma^k r_{k+1},$$
$$\nabla_\theta J
=\mathbb E\!\left[G_0\sum_{t=0}^{T-1}\nabla_\theta\log\pi_\theta(a_t\mid s_t)\right].$$

Every action gets the **same full return $G_0$** because we differentiate the
probability of the entire trajectory and multiply by its scalar return.
The discounting is already inside $G_0$: multiplying these score terms by an
additional $\gamma^t$ would change the expected gradient. The earlier
`reward_to_go_weights` helper uses a different signal, $\gamma^t G_t$.

For a tabular softmax row, the score is

$$\frac{\partial\log\pi(a\mid s)}{\partial\theta_{s,b}}
=\mathbf 1[a=b]-\pi(b\mid s).$$

Other rows have zero derivative for this action. PyTorch computes this with
`autograd.grad` through `log_softmax`; we use the formula only to check it by hand.
[Spinning Up derives the trajectory score identity](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#deriving-the-simplest-policy-gradient)
for an undiscounted return; here we substitute our finite discounted $G_0$.


### Hand-worked trajectory: EAST, then EAST

Continue the `SHG` example, with $\gamma=0.5$ and
$\pi=(0.1,0.2,0.3,0.4)$. The trajectory S → H → G pays $(-6,9)$,
so $G_0=-6+0.5(9)=-1.5$. Its probability is $0.2^2=0.04$.
Both actions have score $(-0.1,0.8,-0.3,-0.4)$ in their own row:

| Parameter row | Score contribution | $G_0$ × score |
| --- | --- | --- |
| $(S,2)$ | $(-0.1,0.8,-0.3,-0.4)$ | $(0.15,-1.2,0.45,0.6)$ |
| $(H,1)$ | $(-0.1,0.8,-0.3,-0.4)$ | $(0.15,-1.2,0.45,0.6)$ |
| Unvisited rows | $(0,0,0,0)$ | $(0,0,0,0)$ |

This **single** trajectory penalizes EAST at H even though EAST immediately
reaches the goal there. Its earlier hazard penalty is included in the shared
return. That irrelevant past reward has zero expected score contribution at H,
but varies across samples and adds noise to its gradient. Independent reward
noise also enters $G_0$ and weights every action score. These observations
motivate later estimators; this bead keeps the full-return signal.


In [12]:
parameters = tuple(tiny_policy.logits.values())
hand_states = [tiny.start_state, State((0, 1), 1)]
score, = torch.autograd.grad(tiny_policy.log_probabilities(hand_states[0])[1],
                            (tiny_policy.logits[hand_states[0]],))
print("EAST score at S,2:", score.cpu().numpy().round(3))
hand = ReinforceEpisode(
    torch.stack([tiny_policy.log_probabilities(s)[1] for s in hand_states]),
    [-6., 9.],
)
hand_gradient = dict(zip(tiny_policy.logits, full_return_gradient([hand], parameters, gamma=.5)))
for state in hand_states:
    print(f"{state}: {hand_gradient[state].cpu().numpy().round(3)}")
    torch.testing.assert_close(hand_gradient[state], score.new_tensor([.15, -1.2, .45, .6]),
                               rtol=1e-12, atol=1e-14)


EAST score at S,2: [-0.1  0.8 -0.3 -0.4]
State(cell=(0, 0), remaining=2): [ 0.15 -1.2   0.45  0.6 ]
State(cell=(0, 1), remaining=1): [ 0.15 -1.2   0.45  0.6 ]


### Average whole episode sums, then check the exact expectation

`algorithms.reinforce.full_return_gradient` takes `ReinforceEpisode` records:
a 1D tensor of selected action log-probabilities, retaining their graph, and
recorded rewards. The caller supplies **complete episodes collected under the
same frozen policy** and the parameters to differentiate. Observations and
action labels stay with the game; this algorithm only needs log-probabilities.

It differentiates the scalar surrogate

$$L(\theta)=\frac1N\sum_{i=1}^N G_0^{(i)}
\underbrace{\sum_t\log\pi_\theta(a_t^{(i)}\mid s_t^{(i)})}_{\text{episode sum}},$$

holding recorded rewards fixed. This returns the **ascent** gradient, in parameter
order, without changing logits or their `.grad` buffers. $L$ is a device for
obtaining the gradient, not an estimate of J. Each call consumes its graph;
reevaluate log-probabilities for the next probe. There is no optimizer step.
Divide by episode count $N$, including for unequal lengths; averaging flattened
transitions would instead divide by the number of actions.

Our no-slip, horizon-2 `SHG` model has exactly $4^2=16$ action sequences.
Enumerate each complete path, compute its probability and sampled gradient,
then sum **probability × gradient**. This deterministic check must recover the
previous Bellman/autograd reference in every parameter row.


In [13]:
from itertools import product

expected_rows = torch.zeros((len(parameters), 4), dtype=torch.float64, device=device)
probability_mass = 0.
for actions in product(Action, repeat=2):
    state, path_probability = tiny.start_state, 1.
    selected_logp, rewards = [], []
    for action in actions:
        index = tuple(Action).index(action)
        path_probability *= tiny_policy.probabilities(state)[index].item()
        selected_logp.append(tiny_policy.log_probabilities(state)[index])
        outcome, = tiny.transitions(state, action)  # This model has no slip.
        rewards.append(outcome.reward)
        state = outcome.next_state
    assert tiny.is_terminal(state)
    record = ReinforceEpisode(torch.stack(selected_logp), rewards)
    path_gradient = torch.stack(full_return_gradient([record], parameters, gamma=.5))
    expected_rows += path_probability * path_gradient
    probability_mass += path_probability

exact_rows = torch.stack([reference.gradient[s] for s in tiny_policy.logits])
assert np.isclose(probability_mass, 1.)
torch.testing.assert_close(expected_rows, exact_rows, rtol=1e-12, atol=1e-14)
print(f"16 paths; probability mass = {probability_mass:.12f}")
print(f"Max |expected REINFORCE - exact gradient| = {(expected_rows - exact_rows).abs().max().item():.2e}")
print("Expected start gradient:", expected_rows[list(tiny_policy.logits).index(tiny.start_state)].cpu().numpy().round(4))


16 paths; probability mass = 1.000000000000
Max |expected REINFORCE - exact gradient| = 1.67e-16
Expected start gradient: [ 0.09 -0.72  0.27  0.36]


### One short seeded sample at the frozen policy

Collect **256 episodes**, seeds **0 through 255**, in that same two-step `SHG`
model, with no slip, $\gamma=0.5$, and independent reward-noise standard
deviation **2.0**. There are 512 environment transitions. This is one batch,
with no training or parameter updates. Log-probabilities and gradient tensors
stay on CUDA; only NumPy sampling and printed values cross to CPU.

Compute each episode's gradient so we can report a standard error, then verify
that the batch API returns their mean. The sampled mean need not equal the
reference. The standard error measures variation in this small sample; it is
not an exact error bound or a guarantee that increasing sample count improves
every realized estimate. The enumeration above establishes agreement in
expectation. This sample illustrates finite-data noise and is not a variance
comparison between estimators.


In [14]:
from time import perf_counter

started = perf_counter()
frozen_logits = {s: row.detach().clone() for s, row in tiny_policy.logits.items()}
seeds, sample_sigma, sample_gamma = list(range(256)), 2., .5
samples = [sample_episode(tiny, tiny_policy, seed=s, reward_noise_std=sample_sigma) for s in seeds]
# Recompute selected log-probabilities under the unchanged collecting policy.
records = [ReinforceEpisode(
    torch.stack([tiny_policy.log_probabilities(step.state)[tuple(Action).index(step.action)]
                 for step in sampled.steps]),
    [step.reward for step in sampled.steps],
) for sampled in samples]
episode_gradients = torch.stack([
    torch.stack(full_return_gradient([record], parameters, gamma=sample_gamma)) for record in records
])
# Fresh graphs are needed because the individual derivative calls consumed them.
batch_records = [ReinforceEpisode(
    torch.stack([tiny_policy.log_probabilities(step.state)[tuple(Action).index(step.action)]
                 for step in sampled.steps]),
    [step.reward for step in sampled.steps],
) for sampled in samples]
sample_mean = torch.stack(full_return_gradient(batch_records, parameters, gamma=sample_gamma))
torch.testing.assert_close(sample_mean, episode_gradients.mean(dim=0), rtol=1e-12, atol=1e-14)
standard_error = episode_gradients.std(dim=0, correction=1) / np.sqrt(len(samples))
start_index = list(tiny_policy.logits).index(tiny.start_state)
print(f"N={len(samples)}, seeds=0..255, transitions={sum(len(e.steps) for e in samples)}, sigma={sample_sigma}")
print("Start row (N,E,S,W):")
print("  exact:    ", exact_rows[start_index].cpu().numpy().round(4))
print("  estimate: ", sample_mean[start_index].cpu().numpy().round(4))
print("  std error:", standard_error[start_index].cpu().numpy().round(4))
print(f"L2 error over all logits: {torch.linalg.vector_norm(sample_mean - exact_rows).item():.4f}")
for state, row in tiny_policy.logits.items():
    torch.testing.assert_close(row, frozen_logits[state], rtol=0, atol=0)
    assert row.grad is None
assert sample_mean.device.type == "cuda"
print(f"Elapsed (collection + individual and batch derivatives): {perf_counter() - started:.3f} s")
print("Policy unchanged; no gradients accumulated; no parameter updates.")


N=256, seeds=0..255, transitions=512, sigma=2.0
Start row (N,E,S,W):
  exact:     [ 0.09 -0.72  0.27  0.36]
  estimate:  [ 0.0014 -0.5436  0.2788  0.2635]
  std error: [0.0705 0.155  0.1037 0.1211]
L2 error over all logits: 0.3922
Elapsed (collection + individual and batch derivatives): 0.232 s
Policy unchanged; no gradients accumulated; no parameter updates.
